# TikHarm policy layer: retrieval, recommended action, case cards, review queue

**Pipeline:** model prediction + evidence (audio tags, speech, on-screen text) -> retrieve the matching policy rule ->
recommended action -> a "case card" for a human moderator, ranked in a priority queue.

**Honest framing (put this in the report):**
- The policy (`prototype_policy.md`) and the retrieval test set (`retrieval_eval_set.csv`) were **written by us**. Retrieval scores show the
  system works as designed. They do not show that its decisions match a real platform's moderators.
- The action table is ours too. Every non-trivial case goes to a human reviewer. Nothing is enforced automatically.
- Suicide-related cases are **escalated to a wellbeing reviewer with support resources**, never silently removed or punished.
- Case cards **hide transcripts and on-screen text for the Adult and Suicide classes**.

**Parts:** (1) parse the policy, (2) retrievers (TF-IDF, dense multilingual, hybrid) and their evaluation on 48 labelled queries,
(3) action table with unit tests, (4) case cards and review-queue metrics on your model's real predictions (needs `predictions_full.csv`
from the fusion notebook, Cell 8), (5) optional labelling sheet for a small real-clip check.

CPU is enough. GPU and Internet only speed up / enable the dense retriever.

In [ ]:
# WORK = "/kaggle/working"

# POLICY_DIR = "/kaggle/input/datasets/sanawerb/actual-policy-files"

# # Read predictions from the existing fusion dataset
# PRED_DIR = "/kaggle/input/datasets/sanawerb/results-fusion3/kaggle/working/results"

# # Keep RES for outputs generated by the policy notebook
# RES = "/kaggle/working/results"
# os.makedirs(RES, exist_ok=True)

# # Read predictions from the attached fusion dataset
# pred_file = f"{PRED_DIR}/predictions_full.csv"

# print("Predictions exist:", os.path.exists(pred_file))
# print("Reading predictions from:", pred_file)
# TEXT_DIR = "/kaggle/input/datasets/sanawerb/text-new/kaggle/working/text"
# PREP = "/kaggle/input/datasets/sanawerb/preparation"

In [6]:
# Cell 1: config and policy parsing
import os, re, json, glob, time
import numpy as np, pandas as pd

WORK = "/kaggle/working"

POLICY_DIR = "/kaggle/input/datasets/sanawerb/actual-policy-files"

# Read predictions from the existing fusion dataset
PRED_DIR = "/kaggle/input/datasets/sanawerb/results-fusion3/kaggle/working/results"

# Keep RES for outputs generated by the policy notebook
RES = "/kaggle/working/results"
os.makedirs(RES, exist_ok=True)

# Read predictions from the attached fusion dataset
pred_file = f"{PRED_DIR}/predictions_full.csv"

print("Predictions exist:", os.path.exists(pred_file))
print("Reading predictions from:", pred_file)
TEXT_DIR = "/kaggle/input/datasets/sanawerb/text-new/kaggle/working/text"
PREP = "/kaggle/input/datasets/sanawerb/preparation"                 # manifest_full.csv (audio tags, optional)
CLASS_NAMES = ["Safe", "Harmful", "Adult", "Suicide"]
SENSITIVE = {"Adult", "Suicide"}
SEV_W = {"low": 1, "medium": 2, "high": 3, "critical": 4}

def parse_policy(path):
    rules, cur = [], None
    for line in open(path, encoding="utf-8"):
        line = line.rstrip("\n")
        m = re.match(r"^### (R-[A-Z]+-\d+) \| (.+)$", line)
        if m:
            cur = dict(rule_id=m.group(1), title=m.group(2).strip(), category=m.group(1).split("-")[1]); rules.append(cur); continue
        m = re.match(r"^(severity|action|support_resources|text|exceptions): (.*)$", line)
        if m and cur is not None: cur[m.group(1)] = m.group(2).strip()
    pol = pd.DataFrame(rules)
    pol["support_resources"] = pol.support_resources.eq("yes")
    pol["doc"] = pol.title + ". " + pol.text + " " + pol.exceptions.fillna("")
    return pol

policy = parse_policy(f"{POLICY_DIR}/prototype_policy.md")
print(len(policy), "rules |", policy.category.value_counts().to_dict())
print(policy[["rule_id", "title", "severity", "action"]].to_string(index=False))
assert policy.rule_id.is_unique and policy.severity.isin(SEV_W).all()

# which policy categories may be retrieved for each predicted class (class-conditioned retrieval)
ALLOWED = {"Safe": {"GEN"}, "Harmful": {"VIO", "HAR", "DAN", "GEN"}, "Adult": {"ADU", "HAR", "GEN"}, "Suicide": {"SUI", "GEN"}}
CLASS_HINT = {"Safe": "", "Harmful": "violence, bullying, harassment or dangerous activity",
              "Adult": "adult or sexual content", "Suicide": "suicide or self-harm"}

Predictions exist: True
Reading predictions from: /kaggle/input/datasets/sanawerb/results-fusion3/kaggle/working/results/predictions_full.csv
31 rules | {'VIO': 7, 'HAR': 6, 'ADU': 6, 'SUI': 5, 'DAN': 4, 'GEN': 3}
 rule_id                                        title severity              action
R-VIO-01                       Real physical violence     high              remove
R-VIO-02                      Graphic injury and gore     high              remove
R-VIO-03                          Threats of violence     high              remove
R-VIO-04                    Glorification of violence     high              remove
R-VIO-05         Weapons displayed or used recklessly   medium        age_restrict
R-VIO-06         Staged, sport and fictional violence      low               allow
R-VIO-07                    Riots and street disorder   medium               limit
R-HAR-01               Insults and degrading language   medium               limit
R-HAR-02    Bullying and humiliation of

In [7]:
# Cell 2: retrievers
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfRetriever:
    def __init__(self, docs):
        self.vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True, lowercase=True).fit(docs)
        self.D = self.vec.transform(docs)
    def scores(self, queries):
        return (self.vec.transform(queries) @ self.D.T).toarray()

def build_dense(docs, name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"):
    import torch
    from transformers import AutoTokenizer, AutoModel
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(name); enc = AutoModel.from_pretrained(name).to(dev).eval()
    def embed(texts, bs=32):
        out = []
        for i in range(0, len(texts), bs):
            x = tok(list(texts[i:i + bs]), padding=True, truncation=True, max_length=256, return_tensors="pt").to(dev)
            with torch.no_grad():
                h = enc(**x).last_hidden_state; m = x["attention_mask"].unsqueeze(-1).float()
                e = torch.nn.functional.normalize((h * m).sum(1) / m.sum(1).clamp(min=1), dim=1)
            out.append(e.cpu().numpy())
        return np.concatenate(out)
    class Dense:
        D = embed(docs)
        def scores(self, queries): return embed(list(queries)) @ self.D.T
    return Dense()

def rrf(score_mats, k=60):
    """reciprocal rank fusion of several (n_queries, n_rules) score matrices"""
    out = np.zeros_like(score_mats[0])
    for S in score_mats:
        ranks = np.argsort(np.argsort(-S, axis=1), axis=1) + 1
        out += 1.0 / (k + ranks)
    return out

DOCS = policy.doc.tolist()
RETR = {"tfidf": TfidfRetriever(DOCS)}
if not os.environ.get("NO_DENSE"):
    try:
        RETR["dense"] = build_dense(DOCS); print("dense retriever ready")
    except Exception as e:
        print("dense retriever unavailable (needs Internet + transformers):", str(e)[:100])

def score_matrix(queries, method):
    if method == "hybrid":
        return rrf([RETR[m].scores(queries) for m in RETR])
    return RETR[method].scores(queries)

METHODS = list(RETR) + (["hybrid"] if len(RETR) > 1 else [])

def rank_rules(queries, pred_classes, method, class_filter=True):
    S = score_matrix(queries, method).copy()
    if class_filter:
        cats = policy.category.values
        for i, c in enumerate(pred_classes):
            S[i, ~np.isin(cats, list(ALLOWED[c]))] = -1e9
    return np.argsort(-S, axis=1), S

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

dense retriever ready


In [8]:
# Cell 3: evaluate retrieval on the 48 labelled queries
ev = pd.read_csv(f"{POLICY_DIR}/retrieval_eval_set.csv")
ev["gold"] = ev.gold_rules.str.split(";")
assert all(g in set(policy.rule_id) for gs in ev.gold for g in gs), "an evaluation gold rule is not in the policy"
ids = policy.rule_id.values

def eval_retrieval(method, class_filter, queries=None):
    order_, _ = rank_rules(ev.evidence_text.tolist(), ev.predicted_class.tolist(), method, class_filter)
    first = np.array([min([list(ids[o]).index(g) + 1 for g in gs if g in ids[o]] or [999]) for o, gs in zip(order_, ev.gold)])
    return first

def summarize(first, nboot=2000):
    rng = np.random.default_rng(0)
    def stat(f): return dict(R1=np.mean(f <= 1), R3=np.mean(f <= 3), R5=np.mean(f <= 5), MRR=np.mean(1.0 / np.minimum(f, 999)))
    pt = stat(first); b = [stat(first[rng.integers(0, len(first), len(first))]) for _ in range(nboot)]
    return {k: f"{pt[k]:.2f} [{np.percentile([x[k] for x in b], 2.5):.2f}, {np.percentile([x[k] for x in b], 97.5):.2f}]" for k in pt}

rows, FIRST = [], {}
for m in METHODS:
    for cf in (False, True):
        f = eval_retrieval(m, cf); FIRST[(m, cf)] = f
        rows.append(dict(retriever=m, class_filter=cf, **summarize(f)))
tab = pd.DataFrame(rows); print(f"retrieval on {len(ev)} labelled queries (a hit = any acceptable rule; 95% bootstrap intervals, small n)")
print(tab.to_string(index=False)); tab.to_csv(f"{RES}/retrieval_eval.csv", index=False)

best = max(FIRST, key=lambda k: (np.mean(FIRST[k] <= 3), np.mean(1 / np.minimum(FIRST[k], 999))))
print("\nbest configuration by Recall@3:", best)
f = FIRST[best]
ev["rank_of_gold"] = f
print("\nby language (Recall@3):", {l: round(float(np.mean(f[ev.language == l] <= 3)), 2) for l in ev.language.unique()})
print("by predicted class (Recall@3):", {c: round(float(np.mean(f[ev.predicted_class == c] <= 3)), 2) for c in ev.predicted_class.unique()})
miss = ev[f > 3]
order_b, _ = rank_rules(ev.evidence_text.tolist(), ev.predicted_class.tolist(), best[0], best[1])
print(f"\nqueries where no gold rule is in the top 3 ({len(miss)}):")
for i in miss.index:
    print(f"  {ev.query_id[i]} gold={ev.gold_rules[i]} top3={[ids[j] for j in order_b[i][:3]]} | {ev.evidence_text[i][:70]}")

retrieval on 48 labelled queries (a hit = any acceptable rule; 95% bootstrap intervals, small n)
retriever  class_filter                R1                R3                R5               MRR
    tfidf         False 0.65 [0.52, 0.77] 0.77 [0.65, 0.88] 0.83 [0.73, 0.94] 0.74 [0.64, 0.84]
    tfidf          True 0.67 [0.54, 0.79] 0.85 [0.75, 0.96] 0.90 [0.81, 0.98] 0.78 [0.69, 0.87]
    dense         False 0.73 [0.60, 0.85] 0.92 [0.83, 0.98] 0.98 [0.94, 1.00] 0.83 [0.74, 0.90]
    dense          True 0.77 [0.65, 0.88] 1.00 [1.00, 1.00] 1.00 [1.00, 1.00] 0.87 [0.80, 0.93]
   hybrid         False 0.77 [0.65, 0.88] 0.90 [0.81, 0.98] 0.94 [0.85, 1.00] 0.84 [0.75, 0.92]
   hybrid          True 0.81 [0.69, 0.92] 0.96 [0.90, 1.00] 0.98 [0.94, 1.00] 0.88 [0.80, 0.95]

best configuration by Recall@3: ('dense', True)

by language (Recall@3): {'en': 1.0, 'vi': 1.0}
by predicted class (Recall@3): {'Harmful': 1.0, 'Adult': 1.0, 'Suicide': 1.0, 'Safe': 1.0}

queries where no gold rule is in the top 3

In [13]:

# Cell 4: action table (ours) and unit tests

def decide(probs, top_rules):
    """probs: dict class -> probability.
    top_rules: list of policy rows (dicts), best first.
    Returns the recommendation.
    """
    cls = max(probs, key=probs.get)
    p = probs[cls]
    srt = sorted(probs.values(), reverse=True)
    margin = srt[0] - srt[1]
    risk = 1.0 - probs["Safe"]

    base = dict(
        pred_class=cls,
        confidence=round(p, 3),
        margin=round(margin, 3),
        risk=round(risk, 3)
    )

    if cls == "Safe":
        if p >= 0.8:
            return dict(
                base,
                action="allow",
                reason="confident Safe prediction",
                needs_human=False,
                support_resources=False,
                priority=0.0
            )

        return dict(
            base,
            action="human_review",
            reason="Safe is the top class but confidence is low",
            needs_human=True,
            support_resources=False,
            priority=risk
        )

    top = top_rules[0] if top_rules else None

    if p < 0.6 or margin < 0.15:
        return dict(
            base,
            action="human_review",
            reason=(
                "low confidence or two classes are close "
                "(R-GEN-03)"
            ),
            needs_human=True,
            support_resources=(cls == "Suicide"),
            priority=risk * (
                SEV_W[top["severity"]] if top else 2
            ),
            rule_id=top["rule_id"] if top else None
        )

    if top is None:
        return dict(
            base,
            action="human_review",
            reason="no policy rule retrieved",
            needs_human=True,
            support_resources=(cls == "Suicide"),
            priority=risk * 2
        )

    act = top["action"]

    if act == "allow":
        # Model flags a class, but the best policy match
        # describes an allowed context.
        act, why = (
            "human_review",
            f"model flags {cls} but the best match "
            f"{top['rule_id']} is an allowed context"
        )
    else:
        why = f"matches {top['rule_id']} ({top['title']})"

    # Adult safeguard: at least age restriction when
    # Adult is predicted and the rule recommends "limit".
    if cls == "Adult" and act == "limit":
        act = "age_restrict"
        why += (
            "; floor applied: Adult predicted, "
            "so at least age_restrict"
        )

    # Suicide safeguard: do not recommend automatic removal.
    if cls == "Suicide" and act == "remove":
        act, why = (
            "escalate_wellbeing",
            why + "; content held for a wellbeing reviewer, "
            "never auto-removed"
        )

    return dict(
        base,
        action=act,
        reason=why,
        needs_human=True,
        support_resources=(
            bool(top["support_resources"]) or cls == "Suicide"
        ),
        priority=risk * SEV_W[top["severity"]],
        rule_id=top["rule_id"],
        severity=top["severity"]
    )


# Policy lookup and probability helper
R = {r["rule_id"]: r for r in policy.to_dict("records")}
P = lambda **k: {c: k.get(c, 0.0) for c in CLASS_NAMES}


# Unit tests

# Confident Safe prediction
assert decide(
    P(Safe=0.95, Harmful=0.05), []
)["action"] == "allow"

# Low-confidence Safe prediction
assert decide(
    P(Safe=0.55, Harmful=0.45), []
)["action"] == "human_review"

# Clear Harmful prediction with a removal rule
assert decide(
    P(Safe=0.1, Harmful=0.85, Adult=0.05),
    [R["R-VIO-01"]]
)["action"] == "remove"

# Close call must go to human review
assert decide(
    P(Safe=0.2, Harmful=0.45, Adult=0.35),
    [R["R-VIO-01"]]
)["action"] == "human_review"

# Suicide: wellbeing escalation with support resources
d = decide(
    P(Safe=0.05, Suicide=0.9, Harmful=0.05),
    [R["R-SUI-02"]]
)
assert (
    d["action"] == "escalate_wellbeing"
    and d["support_resources"]
    and d["needs_human"]
)

# Suicide: allowed context but model flags the content
d = decide(
    P(Safe=0.05, Suicide=0.9, Harmful=0.05),
    [R["R-SUI-04"]]
)
assert (
    d["action"] == "human_review"
    and d["support_resources"]
)

# Adult prediction must be age-restricted
assert decide(
    P(Safe=0.1, Adult=0.88, Harmful=0.02),
    [R["R-ADU-02"]]
)["action"] == "age_restrict"

# Staged violence rule requires review
assert decide(
    P(Safe=0.1, Harmful=0.85),
    [R["R-VIO-06"]]
)["action"] == "human_review"

# Adult safeguard test using R-ADU-06
assert decide(
    P(Safe=0.1, Adult=0.88, Harmful=0.02),
    [R["R-ADU-06"]]
)["action"] == "age_restrict"

# New Suicide safeguard test using R-SUI-01
assert decide(
    P(Safe=0.05, Suicide=0.9, Harmful=0.05),
    [R["R-SUI-01"]]
)["action"] == "escalate_wellbeing"

print("action-table unit tests passed")


action-table unit tests passed


In [14]:

# Cell 5: case cards and the review queue on your model's predictions

def audio_tag_text(s):
    return str(s).replace("|", ", ") if isinstance(s, str) else ""


def build_query(cls, asr, ocr, tags):
    parts = [CLASS_HINT[cls]]

    # Handle missing values from CSV files (NaN/None)
    asr = "" if pd.isna(asr) else str(asr)
    ocr = "" if pd.isna(ocr) else str(ocr)
    tags = "" if pd.isna(tags) else str(tags)

    if asr.strip():
        parts.append("speech: " + asr[:300])
    if ocr.strip():
        parts.append("on-screen text: " + ocr[:200])
    if tags.strip():
        parts.append("audio: " + tags)

    return ". ".join(p for p in parts if p)


def make_cards(pred, model="fused", method=None,
               class_filter=True, topk=3):
    method = method or best[0]
    pc = [f"p_{model}_{c.lower()}" for c in CLASS_NAMES]

    probs = pred[pc].values
    cls_idx = probs.argmax(1)
    cls = [CLASS_NAMES[i] for i in cls_idx]

    queries = [
        build_query(c, a, o, t)
        for c, a, o, t in zip(
            cls, pred.asr_text, pred.ocr_text, pred.audio_tags
        )
    ]

    order_, S = rank_rules(queries, cls, method, class_filter)
    cards = []

    for i in range(len(pred)):
        top = [policy.iloc[j].to_dict()
               for j in order_[i][:topk]]

        dec = decide(
            {c: float(probs[i][k])
             for k, c in enumerate(CLASS_NAMES)},
            top if cls[i] != "Safe" else []
        )

        card = dict(
            uid=pred.uid.iloc[i],
            model=model,
            **dec,
            modalities=dict(
                audio=bool(pred.aud_ok.iloc[i]),
                speech_text=bool(pred.has_asr.iloc[i]),
                onscreen_text=bool(pred.has_ocr.iloc[i])
            ),
            candidate_rules=[
                dict(
                    rule_id=t["rule_id"],
                    title=t["title"],
                    severity=t["severity"],
                    action=t["action"]
                )
                for t in top
            ] if cls[i] != "Safe" else []
        )

        if cls[i] in SENSITIVE:
            card["evidence"] = "hidden (sensitive class)"
        else:
            card["evidence"] = dict(
                audio_tags=(
                    pred.audio_tags.iloc[i]
                    if isinstance(pred.audio_tags.iloc[i], str)
                    else ""
                ),
                speech=str(pred.asr_text.iloc[i])[:200],
                onscreen=str(pred.ocr_text.iloc[i])[:150]
            )

        card["rationale"] = (
            f"Predicted {cls[i]} (p={dec['confidence']}). "
            f"{dec['reason']}. Recommended: {dec['action']}"
            + (" + show support resources"
               if dec["support_resources"] else "")
            + ". Human review required."
            if dec["needs_human"]
            else f"Predicted {cls[i]} (p={dec['confidence']}). "
                 "No action needed."
        )

        cards.append(card)

    return cards


def triage_metrics(pred, model):
    """How many truly non-Safe clips does the model put at the top
    of the review queue?"""
    risk = 1.0 - pred[f"p_{model}_safe"].values
    ns = pred.label.values != 0
    n = len(pred)
    out = {}

    order_ = np.argsort(-risk)

    for frac in (0.10, 0.20, 0.50):
        k = max(int(n * frac), 1)
        top = order_[:k]

        out[f"precision@{int(frac*100)}%"] = round(
            float(ns[top].mean()), 3
        )
        out[f"recall@{int(frac*100)}%"] = round(
            float(ns[top].sum() / ns.sum()), 3
        )
        out[f"suicide_recall@{int(frac*100)}%"] = round(
            float((pred.label.values[top] == 3).sum()
                  / max((pred.label.values == 3).sum(), 1)), 3
        )

    out["base_rate_non_safe"] = round(float(ns.mean()), 3)
    return out


# Read predictions from the configured input path
if os.path.exists(pred_file):
    pred = pd.read_csv(pred_file)

    tt = f"{TEXT_DIR}/text_table_PRIVATE.csv"
    pred["asr_text"], pred["ocr_text"] = "", ""

    if os.path.exists(tt):
        t = pd.read_csv(
            tt, usecols=["uid", "asr_text", "ocr_text"]
        ).fillna("")

        pred = pred.drop(
            columns=["asr_text", "ocr_text"]
        ).merge(t, on="uid", how="left")

        pred[["asr_text", "ocr_text"]] = (
            pred[["asr_text", "ocr_text"]].fillna("")
        )
    else:
        print(
            "text_table_PRIVATE.csv not found: "
            "case cards will use class and audio tags only"
        )

    mf = f"{PREP}/manifest_full (1).csv"
    pred["audio_tags"] = ""

    if not os.path.exists(mf):
        print(
            f"WARNING: {mf} not found, so audio tags are empty "
            "in the cards. Set PREP_DIR to the folder that holds "
            "manifest_full (1).csv."
        )

    if os.path.exists(mf):
        manifest_cols = pd.read_csv(mf, nrows=0).columns

        if "audio_tags" in manifest_cols:
            manifest = pd.read_csv(
                mf, usecols=["uid", "audio_tags"]
            ).fillna({"audio_tags": ""})

            pred = pred.drop(
                columns="audio_tags"
            ).merge(manifest, on="uid", how="left")

    # Final safeguard: avoid NaN values in query text
    for col in ["asr_text", "ocr_text", "audio_tags"]:
        pred[col] = pred[col].fillna("").astype(str)

    print(len(pred), "test clips")

    labels = pred.label.values
    ns = labels != 0
    base = float(ns.mean())
    crit = np.isin(labels, [2, 3])

    def queue_metrics(prio):
        order_ = np.argsort(-prio, kind="stable")
        out = {}

        for frac in (0.10, 0.20, 0.50):
            k = max(int(len(prio) * frac), 1)
            top = order_[:k]
            p = int(frac * 100)

            out[f"precision@{p}% non-Safe"] = round(
                float(ns[top].mean()), 3
            )
            out[
                f"recall@{p}% non-Safe "
                f"(ceiling {min(1.0, frac / base):.2f})"
            ] = round(float(ns[top].sum() / ns.sum()), 3)

            out[f"Adult+Suicide recall@{p}%"] = round(
                float(crit[top].sum() / max(crit.sum(), 1)), 3
            )
            out[f"Suicide recall@{p}%"] = round(
                float((labels[top] == 3).sum()
                      / max((labels == 3).sum(), 1)), 3
            )

        return out

    cards_v = make_cards(pred, "video")
    cards_f = make_cards(pred, "fused")

    rows = [
        dict(
            model="video, severity-weighted priority",
            **queue_metrics(
                np.array([c["priority"] for c in cards_v])
            )
        ),
        dict(
            model="fused, severity-weighted priority",
            **queue_metrics(
                np.array([c["priority"] for c in cards_f])
            )
        ),
        dict(
            model="fused, raw risk only (1 - p_safe)",
            **queue_metrics(1.0 - pred["p_fused_safe"].values)
        )
    ]

    rng = np.random.default_rng(12345)
    rs = [
        queue_metrics(rng.random(len(pred)))
        for _ in range(200)
    ]

    rows.append(
        dict(
            model="random queue (mean of 200)",
            **{
                k: round(float(np.mean([r[k] for r in rs])), 3)
                for k in rs[0]
            }
        )
    )

    qt = pd.DataFrame(rows).set_index("model")

    print(
        f"\nreview-queue quality on {len(pred)} test clips; "
        f"non-Safe base rate = {base:.3f}"
    )
    print(
        "NOTE: this dataset is about 73% non-Safe, far above "
        "real platform prevalence, so precision here is optimistic "
        "and the top of the queue saturates."
    )
    print(qt.T.to_string())
    qt.to_csv(f"{RES}/triage_metrics.csv")
    
    cd_ = pd.DataFrame(cards_f)
    print("\nwhich rule is cited, by predicted class (fused model; empty = none retrieved):")
    print(pd.crosstab(cd_.pred_class, cd_.rule_id.fillna("none")).to_string())
    print("\nwhat each predicted class is sent to (fused model):")
    print(
        pd.crosstab(
            pd.Series(
                [c["pred_class"] for c in cards_f],
                name="predicted"
            ),
            pd.Series(
                [c["action"] for c in cards_f],
                name="action"
            )
        ).to_string()
    )

    cards = cards_f
    acts = pd.Series(
        [c["action"] for c in cards]
    ).value_counts()

    print("\nrecommended actions (fused model):", acts.to_dict())
    print(
        "clips needing human review:",
        sum(c["needs_human"] for c in cards),
        "of", len(cards)
    )

    with open(f"{RES}/case_cards_fused.jsonl", "w") as f:
        for c in cards:
            f.write(
                json.dumps(c, ensure_ascii=False, default=str) + "\n"
            )

    ex = next(
        (
            c for c in cards
            if c["pred_class"] == "Harmful" and c["needs_human"]
        ),
        cards[0]
    )

    print(
        "\nexample card (Harmful class, no sensitive text shown):"
    )
    print(
        json.dumps(ex, indent=1, ensure_ascii=False, default=str)[:900]
    )
    print(
        "\nSaved:", f"{RES}/case_cards_fused.jsonl",
        "(contains transcripts for Safe/Harmful clips: keep it private)"
    )

else:
    print(
        f"{pred_file} not found. Check that PRED_DIR points to "
        "the attached fusion dataset and that predictions_full.csv exists."
    )


870 test clips

review-queue quality on 870 test clips; non-Safe base rate = 0.732
NOTE: this dataset is about 73% non-Safe, far above real platform prevalence, so precision here is optimistic and the top of the queue saturates.
model                               video, severity-weighted priority  fused, severity-weighted priority  fused, raw risk only (1 - p_safe)  random queue (mean of 200)
precision@10% non-Safe                                          1.000                              1.000                              1.000                       0.736
recall@10% non-Safe (ceiling 0.14)                              0.137                              0.137                              0.137                       0.100
Adult+Suicide recall@10%                                        0.187                              0.185                              0.088                       0.101
Suicide recall@10%                                              0.345                              

In [11]:
# Cell 6 (optional): a small labelling sheet for a real-clip retrieval check (Safe and Harmful predictions only)
def labelling_sheet(n=30, seed=0):
    if not os.path.exists(pred_file): print("needs predictions_full.csv"); return
    cards = make_cards(pred, "fused"); df_ = pd.DataFrame(cards)
    sel = df_[(df_.pred_class == "Harmful") & df_.needs_human].sample(min(n, int((df_.pred_class == "Harmful").sum())), random_state=seed)
    sheet = pd.DataFrame(dict(uid=sel.uid, candidate_rules=sel.candidate_rules.map(lambda c: "; ".join(x["rule_id"] for x in c)),
                              recommended=sel.action, gold_rule_ids="", reviewer_notes=""))
    sheet.to_csv(f"{RES}/labelling_sheet_harmful_PRIVATE.csv", index=False)
    print(f"wrote {len(sheet)} rows. Fill gold_rule_ids by reviewing each clip (optional: viewing violent content is your choice, keep it short).")
# labelling_sheet()      # uncomment to create the sheet

In [12]:
labelling_sheet()

wrote 30 rows. Fill gold_rule_ids by reviewing each clip (optional: viewing violent content is your choice, keep it short).


In [13]:
!zip -r results_policy_layer.zip /kaggle/working/results

  adding: kaggle/working/results/ (stored 0%)
  adding: kaggle/working/results/labelling_sheet_harmful_PRIVATE.csv (deflated 72%)
  adding: kaggle/working/results/retrieval_eval.csv (deflated 59%)
  adding: kaggle/working/results/triage_metrics.csv (deflated 53%)
  adding: kaggle/working/results/case_cards_fused.jsonl (deflated 91%)
